# 🌧️ Rainfall Analysis Dashboard
Interactive dashboard built with Panel + hvplot — modelled after the CO2 emissions dashboard.

In [1]:
# Install dependencies if needed
#!pip install panel hvplot pandas numpy

In [2]:
import pandas as pd
import numpy as np
import panel as pn
pn.extension('tabulator')

import hvplot.pandas

## (0) Load & Preprocess Data

In [3]:
# ── Cache data for performance ──────────────────────────────────────────────
if 'rainfall_data' not in pn.state.cache:
    df_raw = pd.read_csv("/Users/nisitasubramani/Desktop/int375 project/rainfall_data.csv")   # ← update path if needed

    # Drop the 'Average' summary row
    df_raw = df_raw[df_raw['S.No'] != 'Average'].copy()

    # Convert numeric columns
    numeric_cols = df_raw.columns[2:]
    df_raw[numeric_cols] = df_raw[numeric_cols].apply(pd.to_numeric, errors='coerce')

    # Friendly short column names (kept as a mapping for display)
    df_raw = df_raw.rename(columns={
        "Total Actual Rainfall (June'17 to May'18) in mm"   : "Total_Actual",
        "Total Normal Rainfall (June'17 to May'18) in mm"   : "Total_Normal",
        "Actual Rainfall in South West Monsoon (June'17 to September'17) in mm" : "SW_Monsoon",
        "Actual Rainfall in North East Monsoon (October'17 to December'17) in mm": "NE_Monsoon",
        "Actual Rainfall in Winter Season (January'18 to and February'18) in mm": "Winter",
        "Actual Rainfall in Hot Weather Season (March'18 to May'18) in mm"      : "Hot_Weather",
    })

    df_raw = df_raw.fillna(0)
    pn.state.cache['rainfall_data'] = df_raw.copy()
else:
    df_raw = pn.state.cache['rainfall_data']

df_raw.head()

,S.No,District,SW_Monsoon,Normal Rainfall in South West Monsoon (June'17 to September'17) in mm,NE_Monsoon,Normal Rainfall in North East Monsoon (October'17 to December'17) in mm,Winter,Normal Rainfall in Winter Season (January'18 to and February'18) in mm,Hot_Weather,Normal Rainfall in Hot Weather Season (March'18 to May'18) in mm,Total_Actual,Total_Normal
0,1,Chennai,449.7,439.1,937.8,789.9,3.1,36.7,2.9,58.5,1393.5,1324.2
1,2,Kancheepuram,488.5,490.8,672.6,641.8,4.4,29.1,14.5,66.0,1180.0,1227.7
2,3,Tiruvallur,498.7,451.6,677.8,589.3,8.1,31.5,17.8,67.2,1202.4,1139.6
3,4,Cuddalore,465.7,383.1,775.9,697.8,55.0,44.1,14.4,81.7,1311.0,1206.7
4,5,Villupuram,441.3,408.3,535.7,499.1,10.5,28.2,24.2,76.0,1011.7,1011.6


In [4]:
# Make the DataFrame interactive (links widgets to data pipelines)
idf = df_raw.interactive()

## (1) Widgets

In [5]:
# ── Top-N slider ─────────────────────────────────────────────────────────────
top_n_slider = pn.widgets.IntSlider(
    name='Top N Districts', start=5, end=30, step=1, value=10
)

# ── Radio buttons: which rainfall metric to rank by ──────────────────────────
yaxis_bar = pn.widgets.RadioButtonGroup(
    name='Rank by',
    options=['Total_Actual', 'Total_Normal', 'SW_Monsoon', 'NE_Monsoon'],
    button_type='success'
)

# ── Season selector for seasonal breakdown ───────────────────────────────────
season_select = pn.widgets.Select(
    name='Season',
    options=['SW_Monsoon', 'NE_Monsoon', 'Winter', 'Hot_Weather'],
    value='SW_Monsoon'
)

# ── Deficit threshold slider (Actual < Normal by X%) ─────────────────────────
deficit_slider = pn.widgets.FloatSlider(
    name='Deficit threshold (%)', start=0, end=80, step=5, value=20
)

## (2) Top-N Districts Bar Chart

In [7]:
top_n_pipeline = (
    idf
    .sort_values(by=yaxis_bar, ascending=False)
    .head(top_n_slider)
    .reset_index(drop=True)
)

top_n_bar = top_n_pipeline.hvplot(
    kind='bar',
    x='District',
    y=yaxis_bar,
    title='Top N Districts by Rainfall',
    xlabel='District',
    ylabel='Rainfall (mm)',
    rot=45,
    height=400,
    width=700,
    color='#4a90d9'
)
top_n_bar

## (3) Tabulator — Full District Data

In [8]:
table_pipeline = (
    idf
    .sort_values(by=yaxis_bar, ascending=False)
    .reset_index(drop=True)
)

rainfall_table = table_pipeline.pipe(
    pn.widgets.Tabulator,
    pagination='remote',
    page_size=10,
    sizing_mode='stretch_width'
)
rainfall_table

## (4) Actual vs Normal Scatterplot

In [9]:
scatter_pipeline = (
    idf[['District', 'Total_Actual', 'Total_Normal']]
    .reset_index(drop=True)
)

scatter_plot = scatter_pipeline.hvplot(
    kind='scatter',
    x='Total_Normal',
    y='Total_Actual',
    by='District',
    size=80,
    alpha=0.7,
    legend=False,
    title='Actual vs Normal Rainfall (per district)',
    xlabel='Normal Rainfall (mm)',
    ylabel='Actual Rainfall (mm)',
    height=450,
    width=550
)
scatter_plot

## (5) Seasonal Rainfall Bar Chart

In [10]:
seasonal_cols = ['SW_Monsoon', 'NE_Monsoon', 'Winter', 'Hot_Weather']

season_pipeline = (
    idf[seasonal_cols + ['District']]
    .sort_values(by=season_select, ascending=False)
    .head(top_n_slider)
    .reset_index(drop=True)
)

season_bar = season_pipeline.hvplot(
    kind='bar',
    x='District',
    y=season_select,
    title='Seasonal Rainfall by District',
    xlabel='District',
    ylabel='Rainfall (mm)',
    rot=45,
    height=400,
    width=600,
    color='#50c878'
)
season_bar

## (6) Deficit Districts Highlight

In [11]:
# Districts where actual rainfall fell short of normal by > threshold %
deficit_pipeline = (
    idf[
        (idf.Total_Normal > 0) &
        (((idf.Total_Normal - idf.Total_Actual) / idf.Total_Normal * 100) >= deficit_slider)
    ][['District', 'Total_Actual', 'Total_Normal']]
    .assign(Deficit_pct=lambda d: ((d.Total_Normal - d.Total_Actual) / d.Total_Normal * 100).round(1))
    .sort_values('Deficit_pct', ascending=False)
    .reset_index(drop=True)
)

deficit_bar = deficit_pipeline.hvplot(
    kind='bar',
    x='District',
    y='Deficit_pct',
    title='Districts with Rainfall Deficit ≥ Threshold',
    xlabel='District',
    ylabel='Deficit (%)',
    rot=45,
    height=400,
    width=600,
    color='#e05c5c'
)
deficit_bar

## (7) Assemble Dashboard with FastListTemplate

In [12]:
template = pn.template.FastListTemplate(
    title='🌧️ India Rainfall Analysis Dashboard',

    # ── Sidebar ──────────────────────────────────────────────────────────────
    sidebar=[
        pn.pane.Markdown("# 🌧️ Rainfall Dashboard"),
        pn.pane.Markdown(
            "Explore district-level rainfall data across seasons.\n\n"
            "Use the controls below to filter and explore."
        ),
        pn.pane.Markdown("---\n## Controls"),
        top_n_slider,
        pn.pane.Markdown("**Rank / colour charts by:**"),
        yaxis_bar,
        pn.pane.Markdown("**Season for seasonal chart:**"),
        season_select,
        pn.pane.Markdown("**Deficit threshold:**"),
        deficit_slider,
    ],

    # ── Main area ────────────────────────────────────────────────────────────
    main=[
        # Row 1: bar chart + table
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### Top Districts by Rainfall"),
                top_n_bar.panel(width=700),
                margin=(0, 25)
            ),
            pn.Column(
                pn.pane.Markdown("### All Districts — Sorted Table"),
                rainfall_table.panel(width=500)
            )
        ),
        # Row 2: scatter + seasonal bar
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### Actual vs Normal Rainfall"),
                scatter_plot.panel(width=550),
                margin=(0, 25)
            ),
            pn.Column(
                pn.pane.Markdown("### Seasonal Rainfall by District"),
                season_bar.panel(width=600)
            )
        ),
        # Row 3: deficit bar — full width
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### Districts with Rainfall Deficit"),
                deficit_bar.panel(width=1200)
            )
        ),
    ],

    accent_base_color="#4a90d9",
    header_background="#4a90d9",
)

template.show()       # opens in a browser tab
template.servable()   # needed when serving via `panel serve`

Launching server at http://localhost:49210


FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-4613948144] Row
        [0] Column(margin=(0, 25))
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=HoloViews, defer_load=False, width=700)
        [1] Column
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=Tabulator, defer_load=False, width=500)
    [main-4613962256] Row
        [0] Column(margin=(0, 25))
            [0] Markdown(str)
            [1] Column(sizing_mode='fixed')
                [0] Column()
                [1] Row(sizing_mode='fixed')
                    [0] HoloViews(NdOverlay, height=450, name='interactive02739', sizing_mode='fixed', width=550)
        [1] Column
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=HoloViews, defer_load=False, width=600)
    [main-4614314640] Row
        [0] Column
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=HoloViews, defer_load=False, width=1200)
    [nav-4612145904] Markdown(str)
    [nav-4612894032] Markdown(str)
    [nav-4612894352] Markdown(str)
    [nav-4598963840] IntSlider(end=30, name='Top N Districts', start=5, value=10)
    [nav-4613096720] Markdown(str)
    [nav-4598964512] RadioButtonGroup(button_type='success', name='Rank by', options=['Total_Actual', ...], value='Total_Actual')
    [nav-4613099456] Markdown(str)
    [nav-4598964176] Select(name='Season', options=['SW_Monsoon', ...], value='SW_Monsoon')
    [nav-4609781648] Markdown(str)
    [nav-4598964848] FloatSlider(end=80, name='Deficit threshold (%)', step=5, value=20)

2026-04-23 21:49:45,801 ERROR: panel.reactive - Callback failed for object named 'Top N Districts' changing properties {'value': 22, 'value_throttled': 22} 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/panel/reactive.py", line 479, in _process_events
    self.param.update(**self_params)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/param/parameterized.py", line 2788, in update
    restore = dict(self_._update(arg, **kwargs))
                   ~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/param/parameterized.py", line 2821, in _update
    self_._batch_call_watchers()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/param/parameterized.py", line 3038, in _batch_call_watchers
    self_._execut